# Transisi Energi Hijau di Indonesia

**Periode:** 17 April – 17 Mei 2026  
**Sumber:** Semantik API (keyword-filtered)  
**Kata kunci:** transisi energi, energi hijau, EBT, energi baru terbarukan, energi surya, geothermal, PLTS, PLTA, biomassa, cofiring, CCS, karbon  

> Semua data volume dan sumber diambil dari endpoint keyword-filtered Semantik API.  
> Data entitas (sentimen, timeline, ko-okurensi) menggunakan endpoint entity-level — scope-nya semua artikel yang menyebut entitas tersebut, bukan hanya artikel dari keyword filter.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = 'data'

def load_json(filename):
    with open(os.path.join(DATA_DIR, filename)) as f:
        return json.load(f)

source_data = load_json('source_comparison.json')
trend_data = load_json('topic_trend.json')
articles = load_json('articles_search.json')

total_articles = sum(s['article_count'] for s in source_data)
print(f'Total artikel: {total_articles} dari {len(source_data)} media')
print(f'Artikel sample: {len(articles)}')
print(f'Periode: {trend_data[0]["date"]} — {trend_data[-1]["date"]}')

## 1. Volume & Sumber Artikel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sources = [s['source'] for s in source_data]
counts = [s['article_count'] for s in source_data]
colors = plt.cm.viridis([i/len(sources) for i in range(len(sources))])

axes[0].barh(sources, counts, color=colors)
axes[0].set_xlabel('Jumlah Artikel')
axes[0].set_title('Artikel per Media')
axes[0].invert_yaxis()

pos = [s['sentiment_distribution']['positive'] for s in source_data]
neg = [s['sentiment_distribution']['negative'] for s in source_data]
neu = [s['sentiment_distribution']['neutral'] for s in source_data]

axes[1].barh(sources, pos, label='Positif', color='#2ecc71')
axes[1].barh(sources, neg, left=pos, label='Negatif', color='#e74c3c')
axes[1].barh(sources, neu, left=[p+n for p,n in zip(pos,neg)], label='Netral', color='#95a5a6')
axes[1].set_xlabel('Jumlah Artikel')
axes[1].set_title('Distribusi Sentimen per Media')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

### Volume Mingguan

In [ ]:
from datetime import datetime
from collections import defaultdict

weekly = defaultdict(int)
for d in trend_data:
    dt = datetime.strptime(d['date'], '%Y-%m-%d')
    week = dt.isocalendar()[1]
    weekly[week] += d['article_count']

weeks = sorted(weekly.keys())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'W{w}' for w in weeks], [weekly[w] for w in weeks], color='#3498db')
for bar, val in zip(bars, [weekly[w] for w in weeks]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha='center', fontweight='bold')
ax.set_ylabel('Jumlah Artikel')
ax.set_title('Volume Artikel per Minggu (Keyword-Filtered)')
plt.tight_layout()
plt.show()

## 2. Top Entities dari Artikel

Entitas yang paling sering muncul di artikel keyword-filtered. Diambil dari `top_entities` pada setiap sumber media di source-comparison.

In [ ]:
from collections import Counter

entity_counts = Counter()
for s in source_data:
    for e in s.get('top_entities', []):
        entity_counts[e['word']] += e['count']

# Show top 20
top20 = entity_counts.most_common(20)
words, counts = zip(*top20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn([c/max(counts) for c in counts])
ax.barh(list(reversed(words)), list(reversed(counts)), color=list(reversed(colors)))
ax.set_xlabel('Jumlah Sebutan')
ax.set_title('Top 20 Entitas dalam Artikel Transisi Energi')
for i, v in enumerate(list(reversed(counts))):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Sentimen Entitas

> **Catatan scope:** Data sentimen diambil dari endpoint entity-level. Artinya, ini mencakup SEMUA artikel yang menyebut entitas tersebut, bukan hanya dari 310 artikel keyword-filtered. Entitas seperti Pertamina dan Prabowo muncul di ribuan artikel lintas topik.

In [ ]:
entities_to_show = ['pertamina', 'pln', 'prabowo', 'bahlil', 'nikel', 'plts']
sentiments = {}
for e in entities_to_show:
    try:
        sentiments[e] = load_json(f'{e}_sentiment.json')
    except FileNotFoundError:
        pass

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, (entity, data) in enumerate(sentiments.items()):
    ax = axes[i//3][i%3]
    labels = ['Positif', 'Negatif', 'Netral']
    values = [data['positive'], data['negative'], data.get('neutral', 0)]
    colors = ['#2ecc71', '#e74c3c', '#95a5a6']
    
    ax.pie(values, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
    ax.set_title(f"{entity.upper()}\nSkor: {data['average_score']:+.2f} | {data['article_count']} sebutan")

plt.suptitle('Distribusi Sentimen per Entitas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Timeline Sebutan Harian

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharex=True)
entity_list = list(sentiments.keys())

for i, entity in enumerate(entity_list):
    timeline = load_json(f'{entity}_timeline.json')
    dates = [d['date'] for d in timeline]
    mentions = [d['mention_count'] for d in timeline]
    
    ax = axes[i//2][i%2]
    color = ['#f39c12', '#3498db', '#9b59b6', '#e74c3c', '#1abc9c', '#e67e22'][i]
    ax.fill_between(range(len(dates)), mentions, alpha=0.2, color=color)
    ax.plot(range(len(dates)), mentions, color=color, linewidth=1.5)
    ax.set_ylabel('Sebutan/hari')
    ax.set_title(entity.upper())
    
    if mentions:
        max_idx = mentions.index(max(mentions))
        ax.annotate(f'{dates[max_idx]}: {mentions[max_idx]}', xy=(max_idx, mentions[max_idx]),
                    xytext=(min(max_idx+3, len(dates)), max(mentions)*0.9),
                    arrowprops=dict(arrowstyle='->', color='gray'), fontsize=8)

tick_pos = list(range(0, len(dates), 5))
for ax in axes[-1]:
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([dates[i] for i in tick_pos], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Ko-okurensi Entitas

Entitas mana yang sering muncul bersamaan — menunjukkan konteks pemberitaan.

In [ ]:
cooc_entities = ['pertamina', 'nikel', 'pln', 'plts']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, entity in enumerate(cooc_entities):
    ax = axes[idx//2][idx%2]
    data = load_json(f'{entity}_cooccurrence.json')
    
    coocs = data['co_occurring_entities'][:8]
    labels = [c['word'] for c in coocs]
    values = [c['co_occurrence_count'] for c in coocs]
    
    colors = plt.cm.RdYlGn([v/max(values) if values else 0 for v in values])
    ax.barh(labels, values, color=colors)
    ax.set_title(f'{entity.upper()} ({data["mention_count"]} sebutan)')
    ax.invert_yaxis()
    
    for j, v in enumerate(values):
        ax.text(v + 0.3, j, str(v), va='center', fontsize=9)

plt.suptitle('Ko-okurensi Entitas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Framing Analisis

Bagaimana media membingkai setiap entitas.

In [ ]:
framing_entities = ['pln', 'plts', 'nikel', 'bahlil']

for entity in framing_entities:
    framing = load_json(f'{entity}_framing.json')
    print(f"\n{'='*60}")
    print(f"FRAMING: {entity.upper()}")
    print('='*60)
    
    if isinstance(framing, dict) and 'by_source' in framing:
        for source, frames in list(framing['by_source'].items())[:4]:
            print(f"\n📰 {source}:")
            for frame in frames[:3]:
                phrase = frame.get('framing_phrase', 'N/A')
                count = frame.get('article_count', 0)
                print(f"  • {phrase} ({count} artikel)")
    elif isinstance(framing, list):
        for frame in framing[:8]:
            print(f"  • {frame['framing_phrase']} ({frame['article_count']} artikel)")

## 7. Temuan Kunci

1. **Volume naik dengan keyword yang lebih luas** — dari 72 artikel (4 keyword) menjadi 310 artikel (12 keyword). Penambahan energi surya, geothermal, PLTS, biomassa, CCS, dan karbon menangkap cakupan yang lebih luas.

2. **Pertamina masih mendominasi** — 307 sebutan, sentimen +2.76. Tapi framing-nya tetap seputar BBM dan harga, bukan transisi hijau.

3. **PLTS adalah entitas energi hijau paling relevan** — 39 sebutan, sentimen +1.79, muncul di 17 hari berbeda. PLTS adalah satu-satunya entitas yang benar-benar terkait langsung dengan energi terbarukan.

4. **PLN negatif karena pemadaman** — sentimen −1.03, framing didominasi krisis listrik Jakarta.

5. **Nikel ambigu** — 49 sebutan, sentimen +1.06. Ko-okurensi menunjukkan nikel terkait dengan batu bara dan ekstraksi, bukan hanya baterai EV.

6. **EBT dan Karbon sangat sedikit** — masing-masing hanya 2-3 sebutan. Topik ini belum masuk pemberitaan mainstream.

### Keterbatasan API

- Endpoint entity-level (`entities/{word}/sentiment`, dll.) tidak mendukung filter `topic_keywords`. Data entitas mencakup semua artikel lintas topik, bukan hanya dari keyword filter kita.
- `articles/search` dibatasi 50 hasil. Untuk analisis yang lebih dalam, perlu pagination atau endpoint bulk.
- `framing/{word}/by-source` untuk Pertamina mengembalikan JSON yang corrupt (embedded newlines dalam framing_phrase). Digunakan `framing/{word}` tanpa by-source.

---
*Data: Semantik API. Metodologi: keyword-filtered endpoints (12 keyword) + entity-level endpoints.*